# Plot2Eq: Обучение модели

### Шаг 1: Настройка окружения и импорты

In [1]:
%load_ext autoreload
%autoreload 2
!pip install 

import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from tqdm.auto import tqdm
import numpy as np
import os
import sys

parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from src.train import train_loop, TrainConfig  # noqa: E402
from src.model import Plot2EqModel  # noqa: E402
from src.tokenizer import Tokenizer  # noqa: E402
from src.dataset import SymbolicDataset  # noqa: E402
from src.datamodule import build_dataloaders  # noqa: E402

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

ERROR: You must give at least one requirement to install (see "pip help install")
PyTorch version: 2.9.1+cpu
CUDA available: False


### Шаг 2: Инспекция архитектуры

Давайте создадим экземпляр модели и посмотрим, сколько в ней обучаемых параметров. Это полезно для оценки требуемых ресурсов и сравнения с другими архитектурами.

In [2]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

cfg = TrainConfig()
tokenizer = Tokenizer()

model = Plot2EqModel(
    vocab_size=len(tokenizer.tokens),
    pad_idx=tokenizer.token_map["<pad>"],
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    num_enc_layers=cfg.num_enc_layers,
    num_dec_layers=cfg.num_dec_layers,
    dropout=cfg.dropout,
)

num_params = count_parameters(model)
print(f"Архитектура модели:\n{model}\n")
print(f"Общее количество обучаемых параметров: {num_params:,}")

Архитектура модели:
Plot2EqModel(
  (cnn_encoder): ConvNeXt1DEncoder(
    (stem): Sequential(
      (0): Conv1d(2, 64, kernel_size=(4,), stride=(2,), padding=(1,))
      (1): GroupNorm(1, 64, eps=1e-05, affine=True)
    )
    (stage1): Sequential(
      (0): ConvNeXt1DBlock(
        (dwconv): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,), groups=64)
        (net): Sequential(
          (0): RMSNorm()
          (1): Linear(in_features=64, out_features=256, bias=True)
          (2): GELU(approximate='none')
          (3): Linear(in_features=256, out_features=64, bias=True)
        )
      )
      (1): ConvNeXt1DBlock(
        (dwconv): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,), groups=64)
        (net): Sequential(
          (0): RMSNorm()
          (1): Linear(in_features=64, out_features=256, bias=True)
          (2): GELU(approximate='none')
          (3): Linear(in_features=256, out_features=64, bias=True)
        )
      )
    )
    (down1): Sequential(

### Шаг 3: Визуализация данных
Убедимся, что наш `SymbolicDataset` и аугментации работают как надо. Посмотрим на несколько примеров из датасета до и после `HandDrawnAugmentation`.

In [ ]:
dataset = SymbolicDataset(data_dir="../data", drawn_augmentation=False)

print("Примеры из датасета без аугментаций:")
dataset.visualize_batch(batch_size=4, apply_transform=False)

print("Те же примеры, но с аугментациями:")
dataset.visualize_batch(batch_size=4, indices=np.arange(4), apply_transform=True)

Загружаем 57 файлов в память (из директории)


 42%|████▏     | 24/57 [00:03<00:04,  7.56it/s]